# Analyzing Flood Impact on Infrastructure

In [5]:
# Import libraries
import ee
import geemap
import pandas as pd
import geopandas as gpd

import os

## . Connect to GEE

In [6]:
# Authenticate GEE
ee.Authenticate()

#Initialize GEE
ee.Initialize()

### . Flood Parameters

In [7]:
# Pre-flood date
before_start = "2022-08-01"
before_end   = "2022-09-10"

# Flood date
after_start = "2022-09-14"
after_end   = "2022-11-18"

# Cloud percentage value
cloud_perc = 58

# Radius for speckle filtering (Sentinel-1)
speckle_radius = 50 # Unit is meter



### . Visualization Parameters

In [8]:
# Visualization Parameters for Boundary
vis_params_aoi = {'color': 'red', 'fillColor': '00000000'}

# Visualization Parameters for Sentinel-2 (True Colour - RGB)
vis_params_s2_rgb = {"min": 0.1, "max": 0.6, "bands": ["B4", "B3", "B2"], "gamma": 0.9}

# Visualization Parameters for Sentinel-2 (False colour)
vis_params_s2_fcc = {"min": 0.01, "max": 0.6, "bands": ["B8", "B4", "B3"], "gamma": 0.9}

# Visualization Parameters for Sentinel-1
vis_params_s1_vv = {"min": -20, "max": 5, "bands": ["VV"], "palette": ["black", "white"]}
vis_params_s1_vh = {"min": -20, "max": 5, "bands": ["VH"], "palette": ["black", "white"]}

# Visualization Parameters for GSW Permanent Water
vis_params_perm_water = {'min': 0,'max': 1,'palette': ["cyan"]}

## . Helper Functions

## . Area of Interest (AOI)

In [9]:
# AOI (GeoJSON)
aoi_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [6.853409, 7.676968],
            [6.685181, 7.676968],
            [6.685181, 7.890588],
            [6.853409, 7.890588],
            [6.853409, 7.676968]
        ]
    ]
}

# Convert to GEE Feature
aoi_feature = ee.Feature(aoi_geojson)


aoi_geometry = aoi_feature.geometry()

# Default location to show the map
map_center_coords = 7.8175, 6.7730


In [10]:
# Show AOI on the map
aoi_map = geemap.Map(center=(7.8175, 6.7730), zoom=10)
aoi_map.add_basemap("SATELLITE")
aoi_map.addLayer(aoi_feature, vis_params_aoi, "AOI Flood")
aoi_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## . Get Admin Boundaries

In [11]:
# Get Nigeria Admin Boundary from FAO
fao_bdry_l0 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0") # Adm0
fao_bdry_l1 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1") # Adm1
fao_bdry_l2 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2") #Adm2

# Get the boundary of Nigeria from FAO GAUL
nga_bdry_l0 = fao_bdry_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # Main Nigerian bdry
nga_bdry_l1 = fao_bdry_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # States bdry
nga_bdry_l2 = fao_bdry_l2.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # LGAs bdry

lokoja_by_name = nga_bdry_l2.filter(ee.Filter.eq("ADM2_NAME", "Lokoja"))

### . Boundary Visualization

In [12]:

# Add Nigeria boundaries
aoi_map.addLayer(nga_bdry_l0, {"color": "black"}, "NGA Bdry L0")
aoi_map.addLayer(nga_bdry_l1, {"color": "blue"}, "NGA State")
aoi_map.addLayer(nga_bdry_l2, {"color": "green"}, "NGA LGA L2")

# Add Lokoja
aoi_map.addLayer(lokoja_by_name, {"color": "black"}, "Lokoja (Name)")

# Display map
aoi_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## . Download and Pre-Process Sentinel-2

In [13]:
# Function to download Sentinel-2
def download_s2(start_date: str, end_date : str, cloud_perc: int, extent : ee.Feature | ee.Geometry):
    """
    Download Sentinel-2 images filtered by date, cloud cover, and geographic extent.

    Args:
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        cloud_perc (int): Maximum allowable cloud pixel percentage.
        extent (ee.Feature|ee.Geometry): The region to filter the collection by.

    Returns:
        ee.ImageCollection: The filtered Sentinel-2 collection.
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                    .filterDate(start_date, end_date) # Dates filter
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_perc)) # Cloud filter
                    .filterBounds(extent)) # Boundary filter   

    img_count =  img_count = img_collection.size().getInfo() # Number of image in the collection

    print(f"There are {img_count} images in the Sentinel-2 image collection\n")
    
    return img_collection



# Function to apply cloud masking to Sentinel-2
# See reference: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY
def mask_clouds_s2(image: ee.Image, cloud_thresh: int):
    """
    Apply cloud masking using cloud probability
    Uses the S2_CLOUD_PROBABILITY dataset to mask out cloudy pixels.

    Args:
        image (ee.Image): Sentinel-2 image with cloud probability band
        cloud_thresh (int): Cloud probability threshold (0-100)

    Returns:
        ee.Image: The cloud-masked image.
    """
    clear_threshold = cloud_thresh
    clouds = ee.Image(
            image.get("cloud_mask")
            ).select("probability")
    
    no_clouds = clouds.lt(clear_threshold)
    masked_image = image.updateMask(no_clouds).copyProperties(image, ["system:time_start"])
    
    return masked_image
    

# Masks from the 20m and 60m bands for bad data at scene edges
def mask_edges_s2(image : ee.Image):
    """
    Mask bad data at scene edges using 20m and 60m band masks.
    
    Args:
        image: ee.Image - Sentinel-2 image
    
    Returns:
        ee.Image - Image with edges masked
    """
    masked_image = image.updateMask(
                    image.select("B8A").mask().updateMask(image.select("B9").mask())
                    ).copyProperties(image, ["system:time_start"])
    
    return masked_image 


# Function to convert Sentinel-2 DN to reflectance
def scale_bands_s2(image: ee.Image):
        """
        Convert Sentinel-2 digital numbers to surface reflectance.
    
    Args:
        image: ee.Image - Sentinel-2 image with DN values
    
    Returns:
        ee.Image - Image with scaled reflectance values (0-1)
        """
        scaled_image = image.multiply(0.0001).copyProperties(image, ["system:time_start"]) # Multiply image by 0.0001 and copy "system:time_start" property 
        return scaled_image


# Function to resample bands in Sentinel-2 to 10m
def resample_bands_s2(image: ee.Image):
        """
        Resample 20m bands to 10m resolution using bilinear interpolation.
    
    Args:
        image: ee.Image - Sentinel-2 image with 10m and 20m bands
    
    Returns:
        ee.Image - Image with all bands at 10m resolution
        """
        bands_10m = image.select(["B2", "B3", "B4", "B8"]) # Select 10m bands
        bands_20m = image.select(["B5", "B6", "B7", "B8A", "B11", "B12"]) # Select 20m bands 
        resampled_image = bands_20m.resample("bilinear").reproject(
                                                            crs = bands_10m.projection(), 
                                                             scale=10)
        
        combined_10m = bands_10m.addBands(resampled_image) # Combine original 10m with 20m
        
        return combined_10m.copyProperties(image, ["system:time_start"])


# Function to compute spectral indices
def compute_indices_s2(image:ee.Image):
        """
        Compute spectral indices (NDVI, NDWI, MNDWI, EVI) for Sentinel-2 imagery.
    
    Args:
        image: ee.Image - Sentinel-2 image with required bands
    
    Returns:
        ee.Image - Original image with added spectral indices
        """
        
        ndvi = image.normalizedDifference(["B8", "B4"]).rename("ndvi")
        ndwi = image.normalizedDifference(["B3", "B8"]).rename("ndwi")
        mndwi = image.normalizedDifference(["B3", "B11"]).rename("mndwi")
        evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            "NIR":image.select("B8"),
            "RED": image.select("B4"),
            "BLUE": image.select("B2")
        }
        ).rename("evi")
        
        updated_image = image.addBands([ndvi, ndwi, mndwi, evi]).copyProperties(image, ["system:time_start"])
        
        return updated_image 



In [14]:
# Download Sentinel-2 images
s2_img_col = download_s2(after_start, after_end, cloud_perc, aoi_geometry)

# Apply cloud masking to ALL IMAGES in the Sentinel-2 collection
# Cloud mask is typically applied using the cloud probability dataset
s2_img_col = s2_img_col.map(lambda img: mask_edges_s2(img))  # Edge masking

# Resample bands for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: resample_bands_s2(img))

# Rescale pixel values  for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: scale_bands_s2(img))
                            
# Compute Indices for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: compute_indices_s2(img))

There are 15 images in the Sentinel-2 image collection



In [15]:
# Convert Sentinel-2 images to a List
s2_img_list = s2_img_col.toList(s2_img_col.size())

In [16]:
# Create a median composite for visualization
s2_composite = s2_img_col.median().clip(aoi_feature)

# Visualize Sentinel-2 imagery

s2_map = geemap.Map(center= map_center_coords, zoom=11)
s2_map.add_basemap("SATELLITE")

s2_map.addLayer(s2_composite, vis_params_s2_rgb, "Sentinel-2 RGB (Post-flood)")
s2_map.addLayer(s2_composite, vis_params_s2_fcc, "Sentinel-2 False Color (Post-flood)")
s2_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
s2_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## . Download and Pre-process Sentinel-1

In [17]:
# Function to download Sentinel-1
def download_s1(start_date: str, end_date: str, extent: ee.Feature | ee.Geometry):
    """
    Download Sentinel-1 GRD images filtered by date, instrument mode, resolution, and geographic extent.
    
    Args:
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        extent (ee.Feature|ee.Geometry): The region to filter the collection by.
        
    Returns:
        ee.ImageCollection: The filtered Sentinel-1 collection.
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                        .filterDate(start_date, end_date) # Dates filter
                        .filter(ee.Filter.eq("instrumentMode", "IW")) # InstrumentMode
                        .filterMetadata("resolution_meters", "equals", 10) # Resolution filter
                        .filterBounds(extent) # Boundary filter
                        .select(["VV", "VH"])) # Select polarizations

    # Number of images in the Sentinel-1 Collection
    img_count = img_collection.size().getInfo()
    
    print(f"There are {img_count} Sentinel-1 Images")

    return img_collection


# Function to apply Speckle filter to Sentinel-1 (focal median)
def focal_speckle_filter(image : ee.Image, radius : int,  unit: str = "meters"):
    """
    Focal Median filter with circle kernel.
    Apply focal median speckle filter to Sentinel-1 imagery using a circular kernel.
    
    Focal median filtering reduces speckle noise in SAR imagery by replacing
    each pixel with the median value of its neighboring pixels.
    
    Args:
        image (ee.Image): Sentinel-1 image to filter.
        radius (int): Radius of the circular kernel in specified units.
        unit (str): Units for the radius ('meters' or 'pixels'). Default is 'meters'.
        
    Returns:
        ee.Image: Filtered image with reduced speckle noise.
    """
    image_filtered = image.focal_median(radius, "circle", unit)
    image_filtered.copyProperties(image, image.propertyNames())

    return image_filtered

In [18]:
# Download Sentinel-1 imagery
s1_img_col = download_s1(after_start, after_end, aoi_geometry)

# Apply Speckle filter to ALL IMAGES in the Sentinel-1 collection
s1_img_col = s1_img_col.map(lambda img: focal_speckle_filter(img, speckle_radius, "meters"))


There are 4 Sentinel-1 Images


In [19]:
# Convert Sentinel-1 images to a List
s1_img_list = s1_img_col.toList(s1_img_col.size())

In [20]:
# Create a median composite for visualization
s1_composite = s1_img_col.median().clip(aoi_feature)

# Visualize Sentinel-1 imagery
s1_map = geemap.Map(center=map_center_coords, zoom=11)
s1_map.add_basemap("SATELLITE")

s1_map.addLayer(s1_composite, vis_params_s1_vv, "Sentinel-1 VV (Post-flood)")
s1_map.addLayer(s1_composite, vis_params_s1_vh, "Sentinel-1 VH (Post-flood)")
s1_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
s1_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…


## . Download Landsat-8

In [21]:
# Function to download Landsat-8
def download_l8(start_date: str, end_date: str, cloud_perc: int, extent: ee.Feature | ee.Geometry):
    """
    Download Landsat-8 Collection 2 Level 2 imagery filtered by date, cloud cover, and area.

    Args:
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        cloud_perc (int): Maximum allowed cloud cover percentage.
        extent (ee.Feature | ee.Geometry): Area of interest.

    Returns:
        ee.ImageCollection: Filtered Landsat-8 image collection.
    """
    img_collection = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUD_COVER", cloud_perc))
        .filterBounds(extent)
    )

    img_count = img_collection.size().getInfo()
    print(f"There are {img_count} images in the Landsat-8 image collection\n")

    return img_collection


# Band scaling factors for Landsat-8
def scale_bands_l8(image: ee.Image):
    """
    Apply scale factors to Landsat-8 optical and thermal bands.

    Args:
        image (ee.Image): Landsat-8 image.

    Returns:
        ee.Image: Scaled Landsat-8 image.
    """
    optical_bands = image.select(["SR_B.*"]).multiply(0.0000275).add(-0.2)
    thermal_bands = image.select(["ST_B.*"]).multiply(0.00341802).add(149.0)

    scaled_bands = image.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

    return scaled_bands.copyProperties(image, ["system:time_start"])


# Cloud Masking for Landsat-8
def mask_clouds_l8(image: ee.Image):
    """
    Mask clouds and cloud shadows in Landsat-8 SR imagery using the QA_PIXEL band.

    Args:
        image (ee.Image): Landsat-8 image.

    Returns:
        ee.Image: Cloud-masked image.
    """
    qa = image.select("QA_PIXEL")
    cloud = qa.bitwiseAnd(1 << 3).eq(0)
    cloud_shadow = qa.bitwiseAnd(1 << 4).eq(0)
    qa_mask = cloud.And(cloud_shadow)

    masked_image = image.updateMask(qa_mask).copyProperties(image, ["system:time_start"])

    return masked_image


# Function to compute spectral indices
def compute_indices_l8(image: ee.Image):
    """
    Compute spectral indices (NDVI, NDWI, MNDWI, EVI) for Landsat-8 imagery.

    Args:
        image (ee.Image): Landsat-8 image with required bands.

    Returns:
        ee.Image: Original image with added spectral index bands.
    """
    ndvi = image.normalizedDifference(["SR_B5", "SR_B4"]).rename("ndvi")
    ndwi = image.normalizedDifference(["SR_B3", "SR_B5"]).rename("ndwi")
    mndwi = image.normalizedDifference(["SR_B3", "SR_B6"]).rename("mndwi")

    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            "NIR": image.select("SR_B5"),
            "RED": image.select("SR_B4"),
            "BLUE": image.select("SR_B2")
        }
    ).rename("evi")

    updated_image = image.addBands([ndvi, ndwi, mndwi, evi]).copyProperties(
        image, ["system:time_start"]
    )

    return updated_image


In [22]:
# Download Landsat-8 images
l8_img_col = download_l8(after_start, after_end, cloud_perc, aoi_geometry)

# Apply cloud masking to ALL IMAGES in the Landsat-8 collection
l8_img_col = l8_img_col.map(lambda img: mask_clouds_l8(img))

# Rescale pixel values for ALL IMAGES in the Landsat-8 collection
l8_img_col = l8_img_col.map(lambda img: scale_bands_l8(img))

# Clip ALL IMAGES in the Landsat-8 collection to the AOI
l8_img_col = l8_img_col.map(lambda img: img.clip(aoi_geometry))

# Compute Indices for ALL IMAGES in the Landsat-8 collection
l8_img_col = l8_img_col.map(lambda img: compute_indices_l8(img))

There are 5 images in the Landsat-8 image collection



In [23]:
# Convert Landsat-8 images to a List 
l8_img_list = l8_img_col.toList(10)

# Create a median composite for visualization
l8_composite = l8_img_col.median().clip(aoi_geometry)

# Visualization parameters
vis_params_l8_rgb = {
    "bands": ["SR_B4", "SR_B3", "SR_B2"],
    "min": 0,
    "max": 0.3
}

vis_params_l8_fcc = {
    "bands": ["SR_B5", "SR_B4", "SR_B3"],
    "min": 0,
    "max": 0.3
}

# Create map
l8_map = geemap.Map()
l8_map.centerObject(aoi_geometry, 11)

# Optional basemap
l8_map.add_basemap("SATELLITE")

# Add layers
l8_map.addLayer(l8_composite, vis_params_l8_rgb, "Landsat-8 RGB (Post-flood)")
l8_map.addLayer(l8_composite, vis_params_l8_fcc, "Landsat-8 False Color (Post-flood)")
l8_map.addLayer(aoi_geometry, {"color": "red"}, "AOI")

l8_map

Map(center=[7.783777212404713, 6.769295000001247], controls=(WidgetControl(options=['position', 'transparent_b…

## . Flood Mapping

### . Histogram Analysis

In [24]:
import geemap.chart as chart

# Image Histogram for Sentinel-2

# Extract the NDWI band from Sentinel 2 composite
ndwi = s2_composite.select('ndwi')

# Sample values from the NDWI layer
sampled_values_s2 = ndwi.sample(
    region=aoi_geometry, 
    scale=10, 
    numPixels=10000
)

# Options for the histogram plot
hist_options = {
    "title": "NDWI Distribution - Sentinel-2",
    "xlabel": "NDWI Value",
    "ylabel": "Pixel Count",
    "colors": ["#1d6b99"],
}

# Generate Histogram
hist_s2 = chart.feature_histogram(
    features=sampled_values_s2,          
    property="ndwi",         
    maxBuckets=50,                  
    **hist_options                    
)
print(hist_s2)

None


In [25]:
# Image Histogram for Sentinel-1

# S1 VV band
s1_band_vv = s1_composite.bandNames().getInfo()[0] # VV band

# Sample values as features from VV band
sampled_values_s1 = s1_composite.sample(region=aoi_feature.geometry(), 
                            scale=10, 
                            numPixels=10000)

# Options for the histogram plot
hist_options = {
        "title": "Image Distribution",
        "xlabel": "Values",
        "ylabel": "Pixel Count",
        "colors": ["#1d6b99"],
    }

# Distribution (backscatter) values using histogram 


hist_s1 = chart.feature_histogram(
    features=sampled_values_s1,          
    property=s1_band_vv,         
    maxBuckets=50,                  
    **hist_options                    
)

print(hist_s1)


None


### . Flood extraction using Thresholding approach

In [26]:
# Flood extraction function for Sentinel-2

def extract_floodmask_s2(image: ee.Image, s2_ndwi_threshold: float, extent: ee.Feature | ee.Geometry):
    """
    Extract flood mask from Sentinel-2 imagery using NDWI thresholding approach.

    
    Args:
        image (ee.Image): Sentinel-2 image
        s2_ndwi_threshold (float): NDWI threshold for water detection (default 0.0). Higher values detect more water.
        extent (ee.Feature|ee.Geometry): The region to clip by

    Returns:
        ee.Image: Binary flood mask (1 = flood, 0 = non-flood).
    """
    # Create the mask where NDWI is greater than the threshold
    mask = image.select("ndwi").gt(s2_ndwi_threshold).clip(extent).rename("flood_mask")

    # Copy properties to preserve metadata
    mask = mask.copyProperties(image, ["system:time_start"])
    
    # Return the mask, clipped to AOI
    return mask

In [27]:
# Apply the function to extract masks from all imagery Sentinel-2 collection
s2_flood_mask_col = s2_img_col.map(lambda img: extract_floodmask_s2(img, s2_ndwi_threshold=0.1, extent= aoi_geometry))

In [28]:
# Convert Sentinel-2 flood mask collection to a List for visualization
s2_flood_mask_list = s2_flood_mask_col.toList(s2_flood_mask_col.size())
flood_mask_count = s2_flood_mask_col.size().getInfo()

print(f"There are {flood_mask_count} flood masks in the collection")

There are 15 flood masks in the collection


In [29]:
# Visualize Sentinel 2 Flood mask
s2_flood_map = geemap.Map(center=map_center_coords, zoom=11)
s2_flood_map.add_basemap("SATELLITE")

s2_flood_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
s2_flood_map.addLayer(s2_composite, vis_params_s2_fcc, "S2 Scaled Masked Image")

# Loop through all flood masks and add them to the map
for i in range(flood_mask_count):
    mask_img = ee.Image(s2_flood_mask_list.get(i))
    try:
        date = ee.Date(mask_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    except:
        date = f"Mask_{i+1}"
    
    s2_flood_map.addLayer(
        mask_img, 
        {"min": 0, "max": 1, "palette": ["white", "blue"]}, 
        f"S2 Flood Mask - {date}"
    )
    
# Add composite flood mask
s2_flood_composite = s2_flood_mask_col.max()
s2_flood_map.addLayer(
    s2_flood_composite,
    {"min": 0, "max": 1, "palette": ["white", "blue"]},
    "Sentinel-1 Maximum Flood Mask"
)

s2_flood_map


Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [30]:
# Flood extraction function for Sentinel-1

def extract_floodmask_s1(image: ee.Image, s1_threshold: float, polarization: str = "VV", extent: ee.Feature | ee.Geometry = None):
    """
    Extract flood mask from Sentinel-1 imagery using backscatter thresholding.
    
    Flooded areas typically show lower backscatter values (dark) compared to non-flooded areas.
    
    Args:
        image (ee.Image): Sentinel-1 image (should have VV and/or VH bands)
        s1_threshold (float): Backscatter threshold for water detection (in dB).
                               Lower values (more negative) detect more water.
                               Typical range: -25 to -10 dB.
        polarization (str): Polarization band to use ('VV' or 'VH'). Default is 'VV'.
        extent (ee.Feature|ee.Geometry): The region to clip the mask to.
        
    Returns:
        ee.Image: Binary flood mask (1 = flood, 0 = non-flood).
    """
    
    # Create binary mask where backscatter is less than threshold
    flood_mask = image.select(polarization).lt(s1_threshold).clip(extent).rename("flood_mask")
    
    
    # Copy properties to preserve metadata
    flood_mask = flood_mask.copyProperties(image, ["system:time_start"])
    
    return flood_mask

In [31]:
# Apply the function to Sentinel-1 collection
s1_flood_mask_col = s1_img_col.map(lambda img: extract_floodmask_s1(
    img, 
    s1_threshold=-15,  # Adjust threshold based on histogram analysis
    polarization="VV",
    extent=aoi_geometry
))

# Convert to list for visualization
s1_flood_mask_list = s1_flood_mask_col.toList(s1_flood_mask_col.size())
s1_flood_mask_count = s1_flood_mask_col.size().getInfo()
print(f"There are {s1_flood_mask_count} flood masks from Sentinel-1")

There are 4 flood masks from Sentinel-1


In [32]:
# Visualize Sentinel-1 Flood masks
s1_flood_map = geemap.Map(center=map_center_coords, zoom=11)
s1_flood_map.add_basemap("SATELLITE")

s1_flood_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
s1_flood_map.addLayer(s1_composite, vis_params_s1_vv, "Sentinel-1 VV (Filtered)")

# Add individual flood masks
for i in range(s1_flood_mask_count):
    flood_img = ee.Image(s1_flood_mask_list.get(i))
    try:
        date = ee.Date(flood_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    except:
        date = f"Mask_{i+1}"
    
    s1_flood_map.addLayer(
        flood_img,
        {"min": 0, "max": 1, "palette": ["white", "blue"]},
        f"S1 Flood Mask - {date}"
    )

# Add composite flood mask
s1_flood_composite = s1_flood_mask_col.max()
s1_flood_map.addLayer(
    s1_flood_composite,
    {"min": 0, "max": 1, "palette": ["white", "blue"]},
    "Sentinel-1 Maximum Flood Mask"
)

s1_flood_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [33]:
## Flood extraction function for Landsat-8

def extract_floodmask_l8(image: ee.Image, l8_threshold: float, extent: ee.Feature | ee.Geometry):
    """
    Extract flood mask from Landsat-8 imagery using NDWI thresholding approach.

    Args:
        image (ee.Image): Landsat-8 image
        l8_threshold (float): NDWI threshold for water detection

    Returns:
        ee.Image: Binary flood mask (1 = flood, masked elsewhere)
    """
    # Create binary flood mask: 1 = flood, 0 = non-flood
    flood_mask = image.select("ndwi").gt(l8_threshold).rename("flood_mask")

    # Keep only flooded pixels visible and preserve time property
    return flood_mask.updateMask(flood_mask) \
                     .clip(extent) \
                     .copyProperties(image, ["system:time_start"])

In [34]:
# Apply flood mask extraction to all Landsat-8 images
l8_flood_mask_col = l8_img_col.map(lambda img: extract_floodmask_l8(img, l8_threshold=0.1, extent=aoi_geometry))

# Print number of flood masks
l8_flood_mask_count = l8_flood_mask_col.size().getInfo()
print(f"There are {l8_flood_mask_count} flood masks in the collection")

There are 5 flood masks in the collection


In [35]:
# Convert Landsat-8 flood mask collection to a list
l8_flood_mask_list = l8_flood_mask_col.toList(l8_flood_mask_col.size())

In [36]:
# Visualize Landsat-8 Flood mask
l8_flood_map = geemap.Map(center=map_center_coords, zoom=11)
l8_flood_map.add_basemap("SATELLITE")

l8_flood_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
l8_flood_map.addLayer(l8_composite, vis_params_l8_fcc, "L8 Scaled Masked Image")


# Loop through all flood masks and add them to the map
for i in range(l8_flood_mask_count):
    flood_img = ee.Image(l8_flood_mask_list.get(i))
    try:
        date = ee.Date(flood_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    except:
        date = f"Mask_{i+1}"

    l8_flood_map.addLayer(
        flood_img,
        {"min": 0, "max": 1, "palette": ["white", "blue"]},
        f"L8 Flood Mask {date}"
    )

# Calculate and add maximum flood mask
l8_flood_mask_max = l8_flood_mask_col.max()

l8_flood_map.addLayer(
    l8_flood_mask_max,
    {"min": 0, "max": 1, "palette": ["white", "blue"]},
    "L8 Maximum Flood Mask"
)

l8_flood_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## Thematic Layers Downloading

*Download WorldPop Population Data*

In [37]:
# Function to download WorldPop population data 
def download_worldpop_new(year: int, extent: ee.Feature | ee.Geometry):
    """
    Download WorldPop population data for a given year and extent.

    Args:
        year (int): Year of population data (e.g. 2020, 2022)
        extent (ee.Feature | ee.Geometry): Area of interest

    Returns:
        ee.ImageCollection: Filtered WorldPop image collection
    """
    img_collection = (
        ee.ImageCollection("projects/sat-io/open-datasets/WORLDPOP/pop")
        .filter(ee.Filter.calendarRange(year, year, "year"))
        .filterBounds(extent)
    )

    img_count = img_collection.size().getInfo()
    print(f"There are {img_count} WorldPop image(s) for {year}")
    
    return img_collection

In [38]:
# Download 2020 and 2022 WorldPop data
wp_2020_col = download_worldpop_new(2020, aoi_geometry)
wp_2022_col = download_worldpop_new(2022, aoi_geometry)

# Convert collections to images
pop_2020 = wp_2020_col.mosaic().clip(aoi_geometry)
pop_2022 = wp_2022_col.mosaic().clip(aoi_geometry)

There are 3 WorldPop image(s) for 2020
There are 3 WorldPop image(s) for 2022


In [55]:
vis_params_pop = {
    "min": 0,
    "max": 100,
    "palette": ["white", "yellow", "orange", "red", "purple"]
}

pop_map = geemap.Map(center=map_center_coords, zoom=10)
pop_map.add_basemap("SATELLITE")
pop_map.addLayer(pop_2020, vis_params_pop, "WorldPop 2020")
pop_map.addLayer(pop_2022, vis_params_pop, "WorldPop 2022")
pop_map.addLayer(aoi_geometry, {"color": "blue"}, "AOI")
pop_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

*Download ESA World Cover Data*

In [78]:
# Function to download ESA WorldCover land cover data for 2021
def download_worldcover_2021(extent: ee.Feature | ee.Geometry):
    """
    Download ESA WorldCover 2021 land cover data for a given extent.
    
    ESA WorldCover v200 provides 10m resolution global land cover maps for 2021.
    Reference: https://esa-worldcover.org/en
    
    Args:
        extent (ee.Feature | ee.Geometry): Area of interest

    Returns:
        ee.Image: Clipped ESA WorldCover 2021 image with land cover classes
"""
    # Load ESA WorldCover 2021 collection (v200 is the 2021 product)
    img_collection = (
        ee.ImageCollection("ESA/WorldCover/v200")  # v200 = 2021 product
        .filterBounds(extent)
    )
    
    # Get image count for verification
    img_count = img_collection.size().getInfo()
    
    if img_count == 0:
        print("Warning: No WorldCover images found for the specified extent")
        return None
    
    print(f"Found {img_count} ESA WorldCover image(s) for 2021")
    
    # Mosaic and clip to extent
    worldcover_img = (
        img_collection
        .mosaic()  # Combine overlapping tiles
        .clip(extent)  # Clip to AOI
        .copyProperties(img_collection.first(), ["system:time_start"])
    )
    
    return worldcover_img

In [79]:
# Download ESA WorldCover 2021

landcover_2021 = download_worldcover_2021(extent=aoi_geometry)


Found 1 ESA WorldCover image(s) for 2021


In [88]:
# Visualization parameters for ESA WorldCover 2021
vis_params_landcover = {
    "min": 10,
    "max": 100,
    "palette": [
        "#006400",  # 10: Tree cover
        "#ffbb22",  # 20: Shrubland
        "#ffff4c",  # 30: Grassland
        "#f096ff",  # 40: Cropland
        "#fa0000",  # 50: Built-up
        "#b4b4b4",  # 60: Bare / sparse vegetation
        "#f0f0f0",  # 70: Snow and ice
        "#0064c8",  # 80: Permanent water bodies
        "#0096a0",  # 90: Herbaceous wetland
        "#00cf75",  # 95: Mangroves
        "#fae6a0"   # 100: Moss and lichen
    ]
}

# Create thematic map
thematic_map = geemap.Map(center=map_center_coords, zoom=10)
thematic_map.add_basemap("SATELLITE")

# Add layers
thematic_map.addLayer(landcover_2021, vis_params_landcover, "ESA WorldCover 2020")
thematic_map.addLayer(aoi_geometry, {"color": "pink"}, "AOI")

# Display map
thematic_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

### . Download Global Surface Water (GSW) (Permanent water)

In [43]:
def download_gsw(extent: ee.Feature | ee.Geometry, seasonality_threshold: int):
    """
    Download Global Surface Water (GSW) dataset and filter by seasonality.
    
    Args:
        extent (ee.Feature|ee.Geometry): The region to filter the collection by.
        seasonality_threshold (int): Number of months water presence (1-12). 

    Returns:
        ee.Image: The filtered GSW image with water presence layer.
    """
    # Load Global Surface Water dataset
    gsw = (ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
                  .select('seasonality')      # seasonality layer
                  .gte(seasonality_threshold) # Seasonality threshold ranges from 0-12 months
                  .clip(extent)               # Clipped to AOI
    )

    
    print(f"Permanent water loaded")
    
    return gsw


In [44]:
# Download Global Surface Water data
perm_water = download_gsw(aoi_geometry, seasonality_threshold=10)

Permanent water loaded


In [45]:
# Visualize Perm water
s2_flood_map.addLayer(perm_water, vis_params_perm_water, "Permanet Water")
s2_flood_map

Map(bottom=251023.0, center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], p…

### Download SRTM DEM

In [89]:
aoi_coords = [
    [
        [6.853409, 7.676968],
        [6.685181, 7.676968],
        [6.685181, 7.890588],
        [6.853409, 7.890588],
        [6.853409, 7.676968]
    ]
]

aoi_geom = ee.Geometry.Polygon(aoi_coords)

extent = aoi_geom


In [47]:
# Function to download SRTM DEM
def download_srtm(extent: ee.Geometry):
    """
    Download SRTM DEM for a given geographic extent.

    Args:
        extent (ee.Geometry): The region to clip the DEM to.

    Returns:
        ee.Image: The clipped SRTM DEM image.
    """
    dem = ee.Image("USGS/SRTMGL1_003").clip(extent)

    print("SRTM DEM loaded successfully.")

    return dem

In [48]:
srtm_dem = download_srtm(extent)

vis_params_dem = {
    "min": 0,
    "max": 3000,
    "palette": ["blue", "green", "yellow", "brown"]
}

aoi_map.addLayer(srtm_dem, vis_params_dem, "SRTM DEM")
aoi_map

SRTM DEM loaded successfully.


Map(bottom=125662.0, center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], p…

### Download Copernicus DEM GLO-30

In [49]:
# Function to download Copernicus DEM GLO-30
def download_copdem(extent: ee.Geometry):
    """
    Download Copernicus DEM GLO-30 for a given geographic extent.

    Args:
        extent (ee.Geometry): The region to clip the DEM to.

    Returns:
        ee.Image: The clipped Copernicus DEM image.
    """
    dem_collection = (ee.ImageCollection("COPERNICUS/DEM/GLO30")
                      .filterBounds(extent)
                      .select("DEM"))
    
    img_count = dem_collection.size().getInfo()
    print(f"There are {img_count} images in the Copernicus DEM collection\n")
    
    dem = dem_collection.mosaic().clip(extent)
    
    return dem

In [50]:
cop_dem = download_copdem(extent)

vis_params_dem = {
    "min": 0,
    "max": 3000,
    "palette": ["white", "yellow", "orange", "red"]
}

aoi_map.addLayer(cop_dem, vis_params_dem, "Copernicus DEM GLO-30")
aoi_map

There are 1 images in the Copernicus DEM collection



Map(bottom=125662.0, center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], p…

### Download Building Dataset

In [90]:
# Function to download Google Open Building footprints
def download_buildings(extent=  ee.Feature | ee.Geometry):
    """
    Download Google Open Building footprints for a given area of interest.
    
    Args:
        extent: geometry or feature collection
        
    Returns:
        GeoDataFrame: Building footprints as a GeoPandas GeoDataFrame
    """
    # Load and filter building data
    buildings = (ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons")
                 .filterBounds(extent))
    
    # Convert to GeoDataFrame
    buildings_gdf = geemap.ee_to_gdf(buildings)
    
    return buildings_gdf

In [92]:
# Download buildings for your AOI
google_bdg_gdf = download_buildings(extent= aoi_geometry)

# Check the result
print(f"Downloaded {len(google_bdg_gdf)} building footprints")
print(f"Columns: {google_bdg_gdf.columns.tolist()}")
google_bdg_gdf.head()

Downloaded 85253 building footprints
Columns: ['geometry', 'area_in_meters', 'confidence', 'full_plus_code', 'longitude_latitude']


,geometry,area_in_meters,confidence,full_plus_code,longitude_latitude
0,"POLYGON ((6.84367 7.72813, 6.84376 7.72811, 6....",121.089996,0.8750,6FV8PRHV+7F9V,"{'type': 'Point', 'coordinates': [6.8437301966..."
1,"POLYGON ((6.77252 7.70548, 6.77257 7.70548, 6....",33.327900,0.7444,6FV8PQ4F+6238,"{'type': 'Point', 'coordinates': [6.7725495664..."
2,"POLYGON ((6.77857 7.72363, 6.77857 7.72359, 6....",13.429000,0.7444,6FV8PQFH+CCWH,"{'type': 'Point', 'coordinates': [6.7785871893..."
3,"POLYGON ((6.78221 7.72644, 6.78222 7.7264, 6.7...",20.875299,0.8069,6FV8PQGJ+HV9W,"{'type': 'Point', 'coordinates': [6.7822391930..."
4,"POLYGON ((6.78279 7.72679, 6.78281 7.72674, 6....",28.350800,0.8069,6FV8PQGM+P44R,"{'type': 'Point', 'coordinates': [6.7828188762..."


## Helper Functions

*Define a Python GEE Function that generates and Labels Random Points*

In [52]:
def generate_stratified_samples(
    image: ee.Image,
    class_band: str,
    num_points: int,
    region: ee.Geometry,
    scale: int,
    seed: int = 42,
    geometries: bool = True
):
    """
    Generate stratified random sample points and extract pixel values from an image.

    Args:
        image (ee.Image): Input image containing predictor bands and class band.
        class_band (str): Name of the class/label band.
        num_points (int): Number of points per class.
        region (ee.Geometry): AOI for sampling.
        scale (int): Pixel resolution for sampling.
        seed (int): Random seed for reproducibility.
        geometries (bool): Whether to return sample point geometries.

    Returns:
        ee.FeatureCollection: Stratified sample points with extracted pixel values.
    """
    samples = image.stratifiedSample(
        numPoints=num_points,
        classBand=class_band,
        region=region,
        scale=scale,
        seed=seed,
        geometries=geometries
    )
    
    return samples

In [95]:
# Rename flood mask to label
l8_label_mask = l8_flood_mask.rename("label")
print(l8_label_mask.bandNames().getInfo())

# Combine Landsat-8 image and label band
l8_image_mask = l8_composite.addBands(l8_label_mask)
print("Bands in combined image and mask:", l8_image_mask.bandNames().getInfo())

NameError: name 'l8_flood_mask' is not defined

In [ ]:
# Generate stratified random sample points
flood_sample_points_l8 = generate_stratified_samples(
    image=l8_image_mask,
    class_band="label",
    num_points=250,
    region=aoi_geometry,
    scale=30,
    seed=42,
    geometries=True
)

print("Sample size:", flood_sample_points_l8.size().getInfo())
print("Class distribution:", flood_sample_points_l8.aggregate_histogram("label").getInfo())

In [ ]:
# Inspect the samples
print("First sample point:", flood_sample_points_l8.first().getInfo())

In [ ]:
# Visualise the sample points
sample_map_l8 = geemap.Map(center=map_center_coords, zoom=10)
sample_map_l8.add_basemap("SATELLITE")

sample_style = {"color": "blue", "pointSize": 3}
sample_layer_l8 = flood_sample_points_l8.style(**sample_style)

sample_map_l8.addLayer(l8_composite.clip(aoi_geometry), vis_params_l8_fcc, "Landsat-8 Image")
sample_map_l8.addLayer(sample_layer_l8, {}, "Stratified Sample Points")

sample_map_l8

*Define a Python GEE Function that Divides Sample Points to Train/Test*

In [ ]:
# Helper Functions

def split_train_test(samples: ee.FeatureCollection, train_ratio: float = 0.7, seed: int = 42):
    """
    Split sampled points into training and testing sets.

    Args:
        samples (ee.FeatureCollection): Sampled points to split.
        train_ratio (float): Proportion of samples for training set.
        seed (int): Random seed for reproducibility.

    Returns:
        tuple: (train_samples, test_samples)
    """
    split_samples = samples.randomColumn(columnName="random", seed=seed)

    train_samples = split_samples.filter(ee.Filter.lt("random", train_ratio))
    test_samples = split_samples.filter(ee.Filter.gte("random", train_ratio))

    return train_samples, test_samples

In [ ]:
# Combine Landsat-8 image and weak flood mask from thresholding
l8_label_mask = l8_flood_mask.rename("label")
print(l8_label_mask.bandNames().getInfo())

l8_image_mask = l8_composite.addBands(l8_label_mask)
print("Bands in combined image and mask:", l8_image_mask.bandNames().getInfo())

In [ ]:
# Create random points and extract pixel values and flood/non-flood labels
flood_sample_points_l8 = l8_image_mask.stratifiedSample(
    numPoints=250,
    classBand="label",
    region=aoi_geometry,
    scale=30,
    seed=42,
    geometries=True
)

print("Class distribution:", flood_sample_points_l8.aggregate_histogram("label").getInfo())

In [ ]:
# Split into train/test
train_l8, test_l8 = split_train_test(flood_sample_points_l8, train_ratio=0.7, seed=42)

print(f"The training sample size: {train_l8.size().getInfo()}")
print(f"The testing sample size: {test_l8.size().getInfo()}")

In [ ]:
# Inspect them
print(f"The training sample example: {train_l8.first().getInfo()}")
print(f"The testing sample example: {test_l8.first().getInfo()}")

training_classes_l8 = train_l8.aggregate_array("label").distinct()
testing_classes_l8 = test_l8.aggregate_array("label").distinct()

print("Unique Classes in Training Set:", training_classes_l8.getInfo())
print("Unique Classes in Testing Set:", testing_classes_l8.getInfo())

In [ ]:
# Visualize them
ml_flood_map_l8 = geemap.Map(center=map_center_coords, zoom=10)
ml_flood_map_l8.add_basemap("SATELLITE")

train_style = {"color": "blue", "pointSize": 3}
test_style = {"color": "black", "pointSize": 3}

train_layer_l8 = train_l8.style(**train_style)
test_layer_l8 = test_l8.style(**test_style)

ml_flood_map_l8.addLayer(l8_composite.clip(aoi_geometry), vis_params_l8_fcc, "Landsat-8 Image")
ml_flood_map_l8.addLayer(train_layer_l8, {}, "Train Samples")
ml_flood_map_l8.addLayer(test_layer_l8, {}, "Test Samples")

ml_flood_map_l8

## . Damage / Impact Assessment